# Brush penguins → Python rows → another chart

Drag a rectangle in the left chart, then release. Python selects the matching
records and sends them to the independent chart on the right. Click **Clear brush**
to empty the selection. Run all cells once; wait for **Ready**.

This notebook follows the deployed [visualization skill](https://genomespy.app/genome-spy-python/agents/skill.md),
[llms.txt](https://genomespy.app/genome-spy-python/llms.txt), and
[Embed API guide](https://genomespy.app/genome-spy-python/agents/embed-api.md).
It reuses the [brush-linked penguin gallery example](https://genomespy.app/genome-spy-python/gallery/penguin_brush.md):
fields, point styling, species order/colors, and the native count summary.

The gallery links views entirely in the browser. Here the widgets have separate
embeds and dataflows: **brush ranges → Python filtering → target dataset → browser counts**.
No JavaScript connection code is needed.

In [ ]:
import json
import math
from urllib.request import urlopen

import genome_spy as gs

DATA_URL = "https://cdn.jsdelivr.net/npm/vega-datasets@2.9.0/data/penguins.json"
with urlopen(DATA_URL, timeout=30) as response:
    raw_rows = json.load(response)

x_field = "Beak Length (mm)"
y_field = "Beak Depth (mm)"
rows = [
    {"row_id": f"penguin-{i:03d}", **row}
    for i, row in enumerate(raw_rows)
    if all(
        isinstance(row.get(field), (int, float)) and math.isfinite(row[field])
        for field in (x_field, y_field)
    )
]
print(
    f"Plotting {len(rows)} of {len(raw_rows)} rows; excluded {len(raw_rows) - len(rows)} with missing coordinates."
)

In [ ]:
species = ["Chinstrap", "Adelie", "Gentoo"]
colors = gs.Scale(domain=species, range=["#BF5CCA", "#FF6C02", "#0F7574"])
brush = gs.selection_interval(
    "brush", encodings=["x", "y"], on={"type": "mousedown"}, empty=False
)
source_chart = (
    gs.Chart(rows)
    .mark_point(filled=False, size=40, opacity=0.7)
    .encode(
        x=gs.X(f"{x_field}:Q")
        .scale(domain=[30, 60], zero=False)
        .title("Beak length (mm)"),
        y=gs.Y(f"{y_field}:Q")
        .scale(domain=[12, 23], zero=False)
        .title("Beak depth (mm)"),
        color=gs.when(brush)
        .then(gs.Color("Species:N").scale(colors).legend(title="Species"))
        .otherwise(gs.value("lightgrey")),
        tooltip=["row_id:N", "Species:N", f"{x_field}:Q", f"{y_field}:Q"],
    )
    .add_params(brush)
    .properties(width=430, height=300, title="Brush penguins, then release")
)
target_chart = (
    gs.Chart(data={"name": "selected"})
    .transform_aggregate(groupby=["Species"])
    .mark_rect()
    .encode(
        x=gs.X("Species:N")
        .scale(domain=species, padding=0.3)
        .axis(labelAngle=0)
        .title("Species"),
        y=gs.Y("count:Q").scale(domain=[0, 160]).title("Selected penguins"),
        color=gs.Color("Species:N").scale(colors).legend(None),
        tooltip=["Species:N", "count:Q"],
    )
    .properties(
        datasets={"selected": []},
        width=260,
        height=300,
        title="Rows received from Python",
    )
)
source_chart.to_dict()
target_chart.to_dict()

## Connect both widgets

The interval snapshot contains ranges in plotted data units. Python filters both
coordinate columns with native half-open bounds (`lower <= value < upper`)
and retains the original records and snapshot-stable row IDs.
GenomeSpy performs the target chart's count aggregation in its browser dataflow.

Updates happen on release (`delivery="commit"`). Before brushing, after clearing,
or when a brush contains no points, the target is empty. Rerunning setup cancels
the previous listener and closes both old widgets.

In [ ]:
import asyncio
from collections import Counter

import ipywidgets as widgets
from IPython.display import display


def matching_rows(snapshot):
    intervals = snapshot["intervals"]
    x, y = intervals.get("x"), intervals.get("y")
    if not snapshot["active"] or x is None or y is None:
        return []
    return [
        row
        for row in rows
        if min(x) <= row[x_field] < max(x) and min(y) <= row[y_field] < max(y)
    ]


async def connect(previous_task, previous_widgets):
    global source_widget, target_widget, source_api, target_api, selection
    global selected_rows, selected_counts, latest_snapshot, update_count, status
    loop = asyncio.get_running_loop()
    if previous_task is not None:
        previous_task.cancel()
        await asyncio.gather(previous_task, return_exceptions=True)
    for old_widget in previous_widgets:
        old_widget.close()
    selected_rows, selected_counts = [], {}
    latest_snapshot, update_count = None, 0
    source_widget = source_chart.widget(inline=True, controls=False)
    target_widget = target_chart.widget(inline=True, controls=False)
    status = widgets.Label("Connecting…")
    clear_button = widgets.Button(description="Clear brush")
    source_api = target_api = selection = None

    async def clear_selection():
        async with asyncio.timeout(30):
            await selection.clear()

    def clear_clicked(_):
        global clear_task
        clear_task = asyncio.run_coroutine_threadsafe(clear_selection(), loop)

    clear_button.on_click(clear_clicked)
    clear_button.disabled = True
    display(
        widgets.HBox([source_widget, target_widget]),
        widgets.HBox([clear_button, status]),
    )
    stop = None
    try:
        events = asyncio.Queue()
        async with asyncio.timeout(30):
            source_api = await source_widget.get_embed_api()
            target_api = await target_widget.get_embed_api()
            selection = await source_api.params.get_selection("brush")
            stop = await selection.subscribe(events.put_nowait, {"delivery": "commit"})
            events.put_nowait(await selection.get_value())
        while True:
            snapshot = await events.get()
            records = matching_rows(snapshot)
            async with asyncio.timeout(30):
                await target_api.datasets.set("selected", records)
            latest_snapshot = snapshot
            selected_rows = records
            selected_counts = dict(Counter(row["Species"] for row in records))
            update_count += 1
            counts = ", ".join(
                f"{name}: {selected_counts.get(name, 0)}" for name in species
            )
            status.value = f"Ready — {len(records)} rows in Python | {counts}"
            clear_button.disabled = False
    except Exception as error:
        status.value = f"Connection failed: {type(error).__name__}: {error}"
        raise
    finally:
        if stop is not None:
            async with asyncio.timeout(2):
                await stop()


connection_task = asyncio.create_task(
    connect(
        globals().get("connection_task"),
        [
            widget
            for name in ("source_widget", "target_widget")
            if (widget := globals().get(name)) is not None
        ],
    )
)

## Inspect and stop

After brushing, run `selected_rows[:5]` in another cell to inspect the original
records, or `selected_counts` for the per-species counts.

To stop, run `cleanup_task = asyncio.create_task(stop_example())`; inspect
`cleanup_task.result()` when it finishes. A saved HTML chart has no Python
connection; keep this notebook's kernel running for this roundtrip.

In [ ]:
async def stop_example():
    connection_task.cancel()
    try:
        await connection_task
    except asyncio.CancelledError:
        pass
    finally:
        source_widget.close()
        target_widget.close()

## Data use and provenance

Palmer Penguins from the versioned Vega datasets collection, as in the gallery.
Source: Gorman KB, Williams TD, Fraser WR (2014),
[doi:10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081).
Python excludes two rows with missing plotted coordinates, adds IDs from original
source-row positions, and filters by the brush. It performs no statistical analysis.
The species order, palette, point style, and native aggregate pattern are reused;
fixed coordinate/count domains and clearing to an empty target are explicit adaptations.